In [24]:
#3.0 Task
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.pipeline import Pipeline

# Load the cleaned appointment dataset
df = pd.read_csv(r"C:\\Users\\Dipali\\CareLopp_Clinic_Project\\appointments_clean.csv")

# Display the first 5 rows
df.head()

# Check number of rows and columns
print("Dataset shape:", df.shape)

# Check column names
print("\nColumn names:")
print(df.columns.tolist())

# Check missing values
print("\nMissing values:")
print(df.isnull().sum())

# Check data types
print("\nData types:")
print(df.dtypes)

# Count the number of attendees and no-shows
print(df["no_show"].value_counts())

# Calculate the percentage distribution
print(df["no_show"].value_counts(normalize=True) * 100)


Dataset shape: (990, 16)

Column names:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']

Missing values:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64

Data types:
appointment_id      object
patient_id          object
clinic_id           object
booked_on           object
appt_date           object
age                  int64
gender              object
distance_km        float64
fee_inr              int64
booking_channel     object
visit_type          object
reminder_sent       object
prior_no_shows       int64
status       

In [25]:
#Task 3.1 Features, split and baseline

# Step 1: Create input features
X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()

# Step 2: Convert Yes/No into 1/0
X["reminder_sent"] = X["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

# Step 3: One-hot encoding
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"]
)

# Step 4: Create target variable
y = df["no_show"]

# Step 5: Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Step 6: Display dataset sizes
print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

# Step 7: Count no-shows in test set
test_no_shows = (y_test == 1).sum()

print("No-shows in test set:", test_no_shows)

# Step 8: Baseline predictions (always predict attended = 0)
baseline_predictions = [0] * len(y_test)

# Step 9: Calculate baseline accuracy
baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

print("Baseline accuracy:", round(baseline_accuracy, 4))
print("Baseline accuracy (%):", round(baseline_accuracy * 100, 2))




Training size: (792, 11)
Testing size: (198, 11)
No-shows in test set: 49
Baseline accuracy: 0.7525
Baseline accuracy (%): 75.25


In [26]:
#Task 3.2 — Logistic regression
# Step 1: Scale the features

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Step 2: Create and train Logistic Regression
model = LogisticRegression(max_iter=1000)

model.fit(X_train_s, y_train)

# Step 3: Make predictions
y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]

# Step 4: Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob)

# Step 5: Display evaluation results
print("LOGISTIC REGRESSION RESULTS")
print("---------------------------")

print("Accuracy:", round(accuracy, 4))
print("\nConfusion Matrix:")
print(cm)

print("\nPrecision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))
print("ROC-AUC:", round(roc_auc, 4))

# Step 6: Display coefficients
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

print("\nMODEL COEFFICIENTS")
print(coefficients.to_string(index=False))

# Step 7: Calculate reminder odds ratios
reminder_index = list(X.columns).index("reminder_sent")

reminder_coefficient = model.coef_[0][reminder_index]

scaled_odds_ratio = np.exp(reminder_coefficient)

reminder_std = scaler.scale_[reminder_index]

reminder_raw_coefficient = (
    reminder_coefficient / reminder_std
)

reminder_adjusted_odds_ratio = np.exp(
    reminder_raw_coefficient
)

print("\nREMINDER ANALYSIS")
print("-----------------")

print("Reminder coefficient:", reminder_coefficient)

print("Scaled odds ratio:", scaled_odds_ratio)

print("Reminder standard deviation:", reminder_std)

print(
    "Adjusted odds ratio (No to Yes):",
    reminder_adjusted_odds_ratio
)


LOGISTIC REGRESSION RESULTS
---------------------------
Accuracy: 0.7879

Confusion Matrix:
[[139  10]
 [ 32  17]]

Precision: 0.6296
Recall: 0.3469
F1 Score: 0.4474
ROC-AUC: 0.7889

MODEL COEFFICIENTS
                Feature  Coefficient
                    age    -0.253880
            distance_km     0.253448
                fee_inr     0.452761
              lead_days     0.614471
         prior_no_shows     0.649612
          reminder_sent    -0.580669
    booking_channel_App    -0.046804
  booking_channel_Phone     0.215881
booking_channel_Walk-In    -0.186364
visit_type_Consultation     0.015658
   visit_type_Follow-up    -0.015658

REMINDER ANALYSIS
-----------------
Reminder coefficient: -0.5806686033659922
Scaled odds ratio: 0.559524141750829
Reminder standard deviation: 0.4949752621616814
Adjusted odds ratio (No to Yes): 0.3093980903124433


Interpretation:-

The Logistic Regression model achieved an accuracy of 78.79% and a ROC-AUC score of 0.7889 on the test dataset. The model achieved a precision of 62.96%, recall of 34.69%, and F1 score of 44.74%.

The confusion matrix shows that the model correctly identified 17 out of 49 actual no-show appointments, while 32 no-show appointments were incorrectly classified as attendees.

The model identified prior_no_shows and lead_days as positively associated with no-show odds. The reminder_sent feature had a negative coefficient. Its adjusted odds ratio was approximately 0.309, indicating lower estimated no-show odds for appointments where reminders were sent, holding the other model features constant.

Although the model achieved reasonable overall accuracy and ROC-AUC, its recall for no-shows is relatively low. Therefore, it misses a considerable proportion of actual no-show appointments. Further model evaluation and threshold selection may help improve no-show detection.

These findings represent associations in observational data and do not establish causal relationships.

In [27]:
#Task 3.3 — Decision Tree and Overfitting




# --------------------------------
# MODEL 1: UNRESTRICTED DECISION TREE
# --------------------------------

tree_unrestricted = DecisionTreeClassifier(random_state=42)

# Train the model
tree_unrestricted.fit(X_train, y_train)

# Predict on training and test data
train_pred_unrestricted = tree_unrestricted.predict(X_train)
test_pred_unrestricted = tree_unrestricted.predict(X_test)

# Calculate accuracy
train_acc_unrestricted = accuracy_score(y_train, train_pred_unrestricted)
test_acc_unrestricted = accuracy_score(y_test, test_pred_unrestricted)


# --------------------------------
# MODEL 2: DECISION TREE MAX DEPTH 3
# --------------------------------

tree_depth3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

# Train the model
tree_depth3.fit(X_train, y_train)

# Predict on training and test data
train_pred_depth3 = tree_depth3.predict(X_train)
test_pred_depth3 = tree_depth3.predict(X_test)

# Calculate accuracy
train_acc_depth3 = accuracy_score(y_train, train_pred_depth3)
test_acc_depth3 = accuracy_score(y_test, test_pred_depth3)


# --------------------------------
# DISPLAY RESULTS
# --------------------------------

print("DECISION TREE RESULTS")
print("---------------------")

print("\nUnrestricted Decision Tree")
print("Training Accuracy:", round(train_acc_unrestricted, 4))
print("Test Accuracy:", round(test_acc_unrestricted, 4))

print("\nDecision Tree (max_depth=3)")
print("Training Accuracy:", round(train_acc_depth3, 4))
print("Test Accuracy:", round(test_acc_depth3, 4))




DECISION TREE RESULTS
---------------------

Unrestricted Decision Tree
Training Accuracy: 1.0
Test Accuracy: 0.6717

Decision Tree (max_depth=3)
Training Accuracy: 0.822
Test Accuracy: 0.7778


Observation:

The unrestricted Decision Tree achieved 100% training accuracy but only 67.17% test accuracy, indicating significant overfitting. In comparison, limiting the tree depth to 3 reduced training accuracy to 82.20% while improving test accuracy to 77.78%.

    
Conclusion:

The large gap between training and test accuracy in the unrestricted Decision Tree indicates that it has overfitted the training data. Restricting the tree depth to 3 reduces overfitting and improves its ability to generalize to unseen data.

In [28]:
#Task 3.4 — Random Forest and Model Comparison

#Step 1 — Train the Random Forest model
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    roc_auc_score
)

# Create Random Forest model
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train the model
rf.fit(X_train, y_train)

# Predict class labels
rf_pred = rf.predict(X_test)

# Predict probability of no-show (class 1)
rf_prob = rf.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred, zero_division=0)
rf_roc_auc = roc_auc_score(y_test, rf_prob)

# Display results
print("RANDOM FOREST RESULTS")
print("---------------------")
print("Accuracy:", round(rf_accuracy, 4))
print("Recall:", round(rf_recall, 4))
print("ROC-AUC:", round(rf_roc_auc, 4))


#Step 2 — Find the top three important features


# Create a feature importance table
importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
})

# Sort from highest to lowest importance
importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

# Display top three features
print("TOP 3 IMPORTANT FEATURES")
print("-----------------------")
print(importance_df.head(3).to_string(index=False))


#Step 3 — Build the model comparison table


from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

# --------------------------------
# 1. LOGISTIC REGRESSION
# --------------------------------

log_pred = model.predict(X_test_s)
log_prob = model.predict_proba(X_test_s)[:, 1]

log_accuracy = accuracy_score(y_test, log_pred)
log_recall = recall_score(y_test, log_pred, zero_division=0)
log_roc_auc = roc_auc_score(y_test, log_prob)


# --------------------------------
# 2. DECISION TREE (MAX DEPTH 3)
# --------------------------------

dt_pred = tree_depth3.predict(X_test)
dt_prob = tree_depth3.predict_proba(X_test)[:, 1]

dt_accuracy = accuracy_score(y_test, dt_pred)
dt_recall = recall_score(y_test, dt_pred, zero_division=0)
dt_roc_auc = roc_auc_score(y_test, dt_prob)


# --------------------------------
# 3. RANDOM FOREST
# --------------------------------

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred, zero_division=0)
rf_roc_auc = roc_auc_score(y_test, rf_prob)


# --------------------------------
# 4. CREATE COMPARISON TABLE
# --------------------------------

comparison_df = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (max_depth=3)",
        "Random Forest"
    ],
    "Accuracy": [
        log_accuracy,
        dt_accuracy,
        rf_accuracy
    ],
    "Recall": [
        log_recall,
        dt_recall,
        rf_recall
    ],
    "ROC-AUC": [
        log_roc_auc,
        dt_roc_auc,
        rf_roc_auc
    ]
})

print("MODEL COMPARISON TABLE")
print("---------------------")
print(comparison_df.round(4).to_string(index=False))




RANDOM FOREST RESULTS
---------------------
Accuracy: 0.7475
Recall: 0.3061
ROC-AUC: 0.7057
TOP 3 IMPORTANT FEATURES
-----------------------
    Feature  Importance
  lead_days    0.219905
        age    0.196788
distance_km    0.194739
MODEL COMPARISON TABLE
---------------------
                      Model  Accuracy  Recall  ROC-AUC
        Logistic Regression    0.7879  0.3469   0.7889
Decision Tree (max_depth=3)    0.7778  0.2245   0.6997
              Random Forest    0.7475  0.3061   0.7057


Observation:

Based on the comparison, Logistic Regression appears to perform best overall because it achieves the highest accuracy (78.79%), recall (34.69%), and ROC-AUC (0.7889) among the three models.

Since the project focuses on identifying no-show appointments, recall is particularly important. Logistic Regression identifies 34.69% of actual no-shows, compared with 22.45% for the depth-3 Decision Tree and 30.61% for the Random Forest.


Conclusion:

The unrestricted Decision Tree achieved 100% training accuracy but only 67.17% test accuracy, indicating significant overfitting. Restricting the tree depth to 3 reduced overfitting and improved its ability to generalize to unseen data, achieving 77.78% test accuracy.

The Random Forest achieved 74.75% accuracy, 30.61% recall, and a ROC-AUC of 0.7057. Its top three important features were lead_days (0.2199), age (0.1968), and distance_km (0.1947).



----- Final Observation and Conclusion ------

Based on the model comparison, Logistic Regression appears to perform best overall because it achieves the highest accuracy (78.79%), recall (34.69%), and ROC-AUC (0.7889) among the three models.

Since the project focuses on identifying no-show appointments, recall is particularly important. Logistic Regression identifies 34.69% of actual no-shows, compared with 22.45% for the depth-3 Decision Tree and 30.61% for the Random Forest.

Although Logistic Regression performs better than the other models on the reported metrics, its recall is still relatively low. Therefore, it misses a considerable number of actual no-show appointments. Further threshold evaluation may help improve no-show detection.

In [29]:

# ============================================================
# TASK 3.5 — TUNE, CHOOSE MODEL AND THRESHOLD, THEN TEST
# ============================================================

# STEP 0: IMPORT LIBRARIES

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_val_score,
    cross_val_predict
)

# Select the required input features
X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()

# Convert reminder_sent Yes/No into 1/0
X["reminder_sent"] = X["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

# Convert categorical features into dummy variables
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"]
)

# Target variable
y = df["no_show"]

# Split data using the same settings as Task 3.1
# Do not shuffle or sort the original DataFrame before this step.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)


# ============================================================
# PART (a): RANDOM FOREST GRID SEARCH
# ============================================================

print("\nPART (a): RANDOM FOREST GRID SEARCH")
print("-----------------------------------")

# Define parameter combinations
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
}

# Perform GridSearchCV on training data only
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc"
)

# Train and search for the best parameters
grid_search.fit(X_train, y_train)

# Display best parameters and score
print("Best Parameters:", grid_search.best_params_)

print(
    "Best Cross-Validated ROC-AUC:",
    round(grid_search.best_score_, 4)
)


# ============================================================
# PART (b): COMPARE FOUR MODELS USING 5-FOLD CROSS-VALIDATION
# ============================================================

print("\nPART (b): MODEL COMPARISON")
print("-------------------------")

# 1. Logistic Regression pipeline
# Scaler is fitted separately within each fold.

lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

lr_cv_auc = cross_val_score(
    lr_pipe,
    X_train,
    y_train,
    cv=5,
    scoring="roc_auc"
).mean()


# 2. Decision Tree with maximum depth 3
tree_depth3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

tree_cv_auc = cross_val_score(
    tree_depth3,
    X_train,
    y_train,
    cv=5,
    scoring="roc_auc"
).mean()


# 3. Random Forest with 100 trees
rf_100 = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_cv_auc = cross_val_score(
    rf_100,
    X_train,
    y_train,
    cv=5,
    scoring="roc_auc"
).mean()


# 4. Tuned Random Forest
tuned_rf = grid_search.best_estimator_

# GridSearchCV best_score_ is its 5-fold CV ROC-AUC
tuned_rf_cv_auc = grid_search.best_score_


# Create model comparison table
cv_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (max_depth=3)",
        "Random Forest (100 trees)",
        "Tuned Random Forest"
    ],

    "Cross_Validated_ROC_AUC": [
        lr_cv_auc,
        tree_cv_auc,
        rf_cv_auc,
        tuned_rf_cv_auc
    ]
})

# Sort models from highest to lowest ROC-AUC
cv_comparison = cv_comparison.sort_values(
    by="Cross_Validated_ROC_AUC",
    ascending=False
).reset_index(drop=True)

print("\nCROSS-VALIDATED MODEL COMPARISON")
print(cv_comparison.to_string(index=False))


# Select the model with the highest CV ROC-AUC
model_candidates = {
    "Logistic Regression": lr_pipe,
    "Decision Tree (max_depth=3)": tree_depth3,
    "Random Forest (100 trees)": rf_100,
    "Tuned Random Forest": tuned_rf
}

winner_name = cv_comparison.loc[0, "Model"]
winner_model = model_candidates[winner_name]

print("\nSelected Winning Model:", winner_name)


# ============================================================
# PART (c): SELECT THRESHOLD USING OUT-OF-FOLD PREDICTIONS
# ============================================================

print("\nPART (c): THRESHOLD COMPARISON")
print("------------------------------")

# Generate out-of-fold probabilities on training data only
oof_probabilities = cross_val_predict(
    winner_model,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]


# Compare thresholds 0.50 and 0.35
thresholds = [0.50, 0.35]

threshold_results = []

for threshold in thresholds:

    # Convert probabilities into binary predictions
    oof_predictions = (
        oof_probabilities >= threshold
    ).astype(int)

    # Calculate precision
    precision = precision_score(
        y_train,
        oof_predictions,
        zero_division=0
    )

    # Calculate recall
    recall = recall_score(
        y_train,
        oof_predictions,
        zero_division=0
    )

    # Calculate F1 score
    f1 = f1_score(
        y_train,
        oof_predictions,
        zero_division=0
    )

    # Count appointments flagged as high risk
    number_flagged = int(oof_predictions.sum())

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Number_Flagged": number_flagged
    })


# Display threshold comparison
threshold_table = pd.DataFrame(threshold_results)

print("\nTHRESHOLD COMPARISON")
print(threshold_table.to_string(index=False))


# Choose threshold with the higher F1 score
best_threshold_index = threshold_table["F1"].idxmax()

chosen_threshold = threshold_table.loc[
    best_threshold_index,
    "Threshold"
]

print("\nChosen Threshold:", chosen_threshold)


# ============================================================
# PART (d): FIT WINNER AND EVALUATE ON TEST SET ONCE
# ============================================================

print("\nPART (d): FINAL TEST EVALUATION")
print("--------------------------------")

# Fit selected model on the complete training data
winner_model.fit(X_train, y_train)

# Calculate test probabilities
test_probabilities = winner_model.predict_proba(
    X_test
)[:, 1]

# Apply selected threshold
test_predictions = (
    test_probabilities >= chosen_threshold
).astype(int)


# Calculate test precision
test_precision = precision_score(
    y_test,
    test_predictions,
    zero_division=0
)

# Calculate test recall
test_recall = recall_score(
    y_test,
    test_predictions,
    zero_division=0
)

# Calculate test ROC-AUC using probabilities
test_roc_auc = roc_auc_score(
    y_test,
    test_probabilities
)

# Count flagged appointments
test_number_flagged = int(test_predictions.sum())


# Display final test results
print("Winning Model:", winner_name)
print("Chosen Threshold:", chosen_threshold)

print("Test Precision:", round(test_precision, 4))
print("Test Recall:", round(test_recall, 4))
print("Number Flagged:", test_number_flagged)
print("Test ROC-AUC:", round(test_roc_auc, 4))


# ============================================================
# PART (e): SAVE TEST RISK SCORES CSV
# ============================================================

print("\nPART (e): SAVE TEST RISK SCORES")
print("--------------------------------")

# Create a DataFrame with the required four columns
test_risk_scores = pd.DataFrame({

    "appointment_id": df.loc[
        X_test.index, "appointment_id"
    ].to_numpy(),

    "no_show": y_test.to_numpy(),

    "risk_prob": test_probabilities,

    "flag": test_predictions
})


# Save the CSV file
output_path = (
    r"C:\Users\Dipali\CareLopp_Clinic_Project\test_risk_scores.csv"
)

test_risk_scores.to_csv(
    output_path,
    index=False
)


print("CSV file saved successfully!")
print("Saved location:", output_path)

# Display first five rows
print("\nFIRST FIVE ROWS OF TEST RISK SCORES")
print(test_risk_scores.head())

# Check number of rows and columns
print("\nCSV shape:", test_risk_scores.shape)

# Verify required column names
print("CSV columns:", test_risk_scores.columns.tolist())




Training data shape: (792, 11)
Testing data shape: (198, 11)

PART (a): RANDOM FOREST GRID SEARCH
-----------------------------------
Best Parameters: {'max_depth': 3, 'n_estimators': 200}
Best Cross-Validated ROC-AUC: 0.7673

PART (b): MODEL COMPARISON
-------------------------

CROSS-VALIDATED MODEL COMPARISON
                      Model  Cross_Validated_ROC_AUC
        Logistic Regression                 0.773714
        Tuned Random Forest                 0.767324
  Random Forest (100 trees)                 0.746602
Decision Tree (max_depth=3)                 0.678449

Selected Winning Model: Logistic Regression

PART (c): THRESHOLD COMPARISON
------------------------------

THRESHOLD COMPARISON
 Threshold  Precision   Recall       F1  Number_Flagged
      0.50   0.575342 0.248521 0.347107              73
      0.35   0.512987 0.467456 0.489164             154

Chosen Threshold: 0.35

PART (d): FINAL TEST EVALUATION
--------------------------------
Winning Model: Logistic Regressio

Observations:-

(a) Random Forest Hyperparameter Tuning

GridSearchCV was performed using 5-fold cross-validation on the training dataset. The best parameters were:

-----------------------------------
Best Parameters: {'max_depth': 3, 'n_estimators': 200}
Best Cross-Validated ROC-AUC: 0.7673


(b) Model Selection

The four models were compared using 5-fold cross-validation ROC-AUC on the training data.

CROSS-VALIDATED MODEL COMPARISON
                      Model  Cross_Validated_ROC_AUC
        Logistic Regression                 0.773714
        Tuned Random Forest                 0.767324
  Random Forest (100 trees)                 0.746602
Decision Tree (max_depth=3)                 0.678449

Selected Winning Model: Logistic Regression 


(c) Threshold Selection

Out-of-fold training probabilities were used to compare thresholds of 0.50 and 0.35.

Threshold 0.50: Precision = [Value], Recall = [Value], F1 = [Value], Number flagged = [Value].

Threshold 0.35: Precision = [Value], Recall = [Value], F1 = [Value], Number flagged = [Value].

THRESHOLD COMPARISON
 Threshold  Precision   Recall       F1  Number_Flagged
      0.50   0.575342 0.248521 0.347107              73
      0.35   0.512987 0.467456 0.489164             154

Chosen Threshold: 0.35
    

(d) Final Test Evaluation

The selected model was fitted on the complete training dataset and evaluated once on the test dataset using the chosen threshold.

--------------------------------
Winning Model: Logistic Regression
Chosen Threshold: 0.35
Test Precision: 0.5714
Test Recall: 0.5714
Number Flagged: 49
Test ROC-AUC: 0.7889


(e) Risk Score Export

FIRST FIVE ROWS OF TEST RISK SCORES
  appointment_id  no_show  risk_prob  flag
0          A0217        0   0.021565     0
1          A0336        0   0.150917     0
2          A0508        0   0.518692     1
3          A0314        0   0.191300     0
4          A0089        0   0.167718     0

CSV shape: (198, 4)
CSV columns: ['appointment_id', 'no_show', 'risk_prob', 'flag']


Conclusion:-

The final model was selected based on its highest 5-fold cross-validated ROC-AUC,
and its classification threshold was selected using out-of-fold training predictions based on the higher F1 score. 
The final test results provide an independent evaluation of the selected model. In a real clinic, the threshold 
would be chosen according to the number of follow-up calls the front desk can make in an evening, balancing this 
capacity against the cost of an empty appointment slot.
